[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yu-hsiu/QuaCCAToo/blob/main/verify_fix_colab.ipynb)

# 在 Colab 重現 FIG. 4(a):NV-13C 的 Hahn echo ESEEM

論文:L. Tsunaki, A. Singh, S. Trofimov, B. Naydenov, *Digital Twin Simulations Toolbox of the
Nitrogen-Vacancy Center in Diamond*, [arXiv:2507.18759v2](https://arxiv.org/abs/2507.18759)。

本 notebook 用 `Yu-hsiu/QuaCCAToo` 這份 fork,依論文 Sec. III B 的參數重現 FIG. 4(a),
再把模擬頻譜的主峰和只由輸入參數決定的 ESEEM 快頻率比對,作為獨立的定量驗證。

## fork 相對於上游的兩個修正

**1. `PulsedSim.run()` 原本無條件走 QuTiP 的 `parallel_map`。**
工作行程只有在 fork 型平台才會繼承母行程的狀態。Windows / macOS 預設是 spawn,工作行程會重新
import 定義序列的模組,於是 import 之後才設定的 `rho0`、`observable`、`H2`、`c_ops` 在那裡
完全不存在——模擬會安靜地演化另一個物理系統,不會報錯。現在預設改走 `serial_map`,只有明確
給 `map_kw={"num_cpus": N}` 且 `N > 1` 才平行,並在非 fork 平台發出警告。

**2. `num_cpus` 會被裁到本行程真正可用的核心數。**
論文正文寫的是 `hahn_sim.run(map_kw={"num_cpus": 32})`,那是工作站的設定。Colab 容器實際只
排得到 2 個核心,直接開 32 個 worker 只會互相搶同樣兩顆核心,比序列還慢,也可能把 runtime 的
記憶體吃光。修正後會自動降到 `len(os.sched_getaffinity(0))` 並印出警告。

## 0. 安裝

In [ ]:
!git clone -q https://github.com/Yu-hsiu/QuaCCAToo.git
%cd QuaCCAToo
!pip install -q -e .

In [ ]:
import multiprocessing, os, platform, sys

import numpy as np
import matplotlib.pyplot as plt
from qutip import jmat, qeye, tensor

import quaccatoo
from quaccatoo import NV, Hahn

USABLE_CPUS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()

print("python        ", sys.version.split()[0], platform.system())
print("quaccatoo     ", quaccatoo.__version__)
print("start method  ", multiprocessing.get_start_method())
print("os.cpu_count  ", os.cpu_count())
print("usable cores  ", USABLE_CPUS)

## 1. 迴歸測試(約 2 分鐘,可跳過)

預期 `40 passed, 3 skipped`。

In [ ]:
!python -m pytest tests -q

## 2. 系統定義(論文 Sec. III B)

$B_0 = 4.2$ mT,與 NV 軸夾角 $\theta = -45^\circ$,即沿 $\mathbf{z}-\mathbf{x}$ 方向;
$^{14}$N 同位素;溫度 300 K,初始態由 Boltzmann 分布給出。
$^{14}$N 的核自旋不能忽略——它會微幅移動電子自旋能階,改變 Larmor 頻率,進而影響 Hahn echo
的包絡頻率。

In [ ]:
sys1 = NV(
    B0=4.2,
    units_B0="mT",
    theta=-45,
    units_angles="deg",
    N=14,
    temp=300,
    units_temp="K",
)

print("Hilbert space dims:", sys1.rho0.dims)
print("energy levels     :", len(sys1.energy_levels))

耦合 Hamiltonian $\hat{H}_2$ 的超精細張量,單位 MHz:

$$
\mathbf{A}^c_{hf} =
\begin{bmatrix}
  5.0 & -6.3 & -2.9 \\
 -6.3 &  4.2 & -2.3 \\
 -2.9 & -2.3 &  8.2
\end{bmatrix}
$$

最後一項是 $^{13}$C 自身的 Zeeman 項,$\gamma_c = 10.705$ kHz/mT。

In [ ]:
alpha = np.array([[5.0, -6.3, -2.9],
                  [-6.3, 4.2, -2.3],
                  [-2.9, -2.3, 8.2]])

H2 = (
    alpha[0, 0] * tensor(jmat(1, "x"), qeye(3), jmat(1 / 2, "x"))
    + alpha[0, 1] * (tensor(jmat(1, "x"), qeye(3), jmat(1 / 2, "y"))
                     + tensor(jmat(1, "y"), qeye(3), jmat(1 / 2, "x")))
    + alpha[0, 2] * (tensor(jmat(1, "x"), qeye(3), jmat(1 / 2, "z"))
                     + tensor(jmat(1, "z"), qeye(3), jmat(1 / 2, "x")))
    + alpha[1, 1] * tensor(jmat(1, "y"), qeye(3), jmat(1 / 2, "y"))
    + alpha[1, 2] * (tensor(jmat(1, "y"), qeye(3), jmat(1 / 2, "z"))
                     + tensor(jmat(1, "z"), qeye(3), jmat(1 / 2, "y")))
    + alpha[2, 2] * tensor(jmat(1, "z"), qeye(3), jmat(1 / 2, "z"))
    - 10.705e-3 * sys1.B0 * tensor(
        qeye(3), qeye(3),
        np.cos(sys1.theta) * jmat(1 / 2, "z") + np.sin(sys1.theta) * jmat(1 / 2, "x"),
    )
)

sys1.add_spin(H2)
print("dims after adding the 13C:", sys1.rho0.dims)

Rabi 頻率 `w1` 與 MW 激發頻率 `w0`。`w0` 取 $m_S=+1$ 六個核子能階的平均減去 $m_S=0$ 的平均,
這樣短硬脈衝驅動的是 $|m_S=0\rangle \leftrightarrow |m_S=+1\rangle$ 躍遷,與核自旋狀態無關。

In [ ]:
w1 = 15
w0 = np.sum(sys1.energy_levels[12:]) / 6 - np.sum(sys1.energy_levels[1:6]) / 6

print(f"w1 = {w1} MHz")
print(f"w0 = {w0:.4f} MHz")

## 3. Hahn echo 模擬

序列是 $\pi/2 - \tau - \pi - \tau - \pi/2$,$\pi$ 脈衝長度 0.0316 µs。它比 $1/(2\omega_1)$
短,來自超精細增強效應。實際的脈衝間隔是 $\tau$ 減掉 $\pi$ 脈衝長度。

`N_TAU = 2000` 是論文的取樣數。實測 Colab 免費 runtime(2 核心)要 26 分鐘,本機單核 21 分鐘
——瓶頸是每個 τ 點的 ODE 積分,核心少幫助有限。想先看形狀可以改成 200,第 6 節的比對會被跳過。

In [ ]:
N_TAU = 2000

tau_hahn = np.linspace(0.04, 4, N_TAU)

hahn_sim = Hahn(
    free_duration=tau_hahn,
    pi_pulse_duration=0.0316,
    system=sys1,
    h1=w1 * sys1.MW_h1,
    pulse_params={"f_pulse": w0},
    projection_pulse=True,
    time_steps=1000,
)

In [ ]:
%%time
# The paper runs this with num_cpus=32 on a workstation. The value is clamped to the cores
# this runtime may actually use, so the same line is safe to run on Colab.
hahn_sim.run(map_kw={"num_cpus": 32})

results = np.asarray(hahn_sim.results, dtype=float)
print("shape", results.shape, "min %.4f max %.4f" % (results.min(), results.max()))

## 4. FIG. 4(a)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2), dpi=140)
ax.plot(tau_hahn, results, lw=0.8, color="#1f4e79")
ax.set_xlabel(r"$\tau$ ($\mu$s)")
ax.set_ylabel(r"$\mathcal{F}_S$")
ax.set_xlim(0, 4)
ax.set_ylim(0, 1)
ax.set_title("FIG. 4(a) - Hahn echo of the NV-$^{13}$C system")
fig.tight_layout()
plt.show()

## 5. 定量檢查:快調變頻率

論文說這條曲線是 ESEEM,快的調變頻率是 $^{13}$C 在 $|m_S=+1\rangle$ 的 Larmor 頻率,
慢的是在 $|m_S=0\rangle$ 的 Larmor 頻率 $\gamma_c B_0$。

快的那個可以直接對照。$m_S=+1$ 時作用在核自旋上的等效場是超精細張量的第三行加上核 Zeeman,

$$ f_{+1} = \left\| \mathbf{A}^c_{hf}\,\hat{z} + \gamma_c \mathbf{B}_0 \right\|, $$

完全由輸入參數決定,和模擬無關,所以拿它檢查 FFT 的主峰是獨立的驗證。

慢的那個 $\gamma_c B_0 = 45$ kHz 週期是 22 µs,遠長於這裡的 4 µs 視窗
(頻率解析度只有 $1/T = 0.25$ MHz),在這張圖上無法解析,只能看到包絡的形狀。

In [ ]:
dt = tau_hahn[1] - tau_hahn[0]
signal = results - results.mean()
spectrum = np.abs(np.fft.rfft(signal * np.hanning(len(signal)), n=1 << 16))
freqs = np.fft.rfftfreq(1 << 16, dt)

f_fast = freqs[np.argmax(spectrum)]

theta_rad = np.deg2rad(-45)
gamma_c = 10.705e-3  # MHz/mT
B0_vec = gamma_c * 4.2 * np.array([np.sin(theta_rad), 0.0, np.cos(theta_rad)])
f_expected = np.linalg.norm(alpha[:, 2] + B0_vec)

print(f"dominant FFT peak  = {f_fast:.4f} MHz")
print(f"|A[:, 2] + gamma_c B0| = {f_expected:.4f} MHz")
print(f"relative deviation = {abs(f_fast - f_expected) / f_expected:.2%}")
print(f"gamma_c * B0       = {np.linalg.norm(B0_vec) * 1e3:.2f} kHz "
      f"(period {1 / np.linalg.norm(B0_vec):.1f} us, frequency resolution "
      f"{1 / (tau_hahn[-1] - tau_hahn[0]):.3f} MHz)")

assert abs(f_fast - f_expected) / f_expected < 0.02, "fast ESEEM frequency is off"

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.0), dpi=140)
ax.plot(freqs, spectrum / spectrum.max(), lw=0.9, color="#1f4e79")
ax.axvline(f_expected, ls="--", lw=1.0, color="#c00000",
           label=r"$\|\mathbf{A}^c_{hf}\hat{z} + \gamma_c\mathbf{B}_0\|$ = "
                 f"{f_expected:.2f} MHz")
ax.set_xlim(0, 15)
ax.set_xlabel("frequency (MHz)")
ax.set_ylabel("normalized amplitude")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 6. 與參考結果比對

`docs/tutorials/sim_data_tutorials/Hahn_13C.npz` 是同一份程式在 Windows 上跑出來的 2000 點
結果,用來確認之後改動不會悄悄改變這條曲線。只有 `N_TAU = 2000` 時才比得起來。

容差設 1e-4。同一份程式在 Windows 與 Colab(Linux)之間量到的最大差是 3.8e-6,來自 ODE
求解器在不同平台的浮點行為,不是回歸;而真正改到物理的改動會遠大於 1e-4。

In [ ]:
ref_path = "docs/tutorials/sim_data_tutorials/Hahn_13C.npz"

if N_TAU == 2000 and os.path.exists(ref_path):
    ref = np.load(ref_path)
    assert np.allclose(tau_hahn, ref["tau"]), "tau grid does not match the reference"
    max_dev = np.abs(results - ref["results"]).max()
    print(f"max deviation from the reference curve: {max_dev:.2e}")
    assert max_dev < 1e-4, "the simulation no longer reproduces the reference curve"
    print("reproduced")
else:
    print("skipped: needs N_TAU = 2000 and the reference file")

## 7. spawn 行為檢查(選讀)

Colab 是 Linux,預設 fork,所以修正 1 針對的 bug 在這裡本來就不會發作。要看到它,必須強制
子行程用 spawn 啟動,也就是複製 Windows / macOS 的行為。下面的腳本在同一個系統上分別用序列
與平行跑同一組 pseudo-Hadamard RF 相位掃描,論文值是 2.9 rad。

In [ ]:
%%writefile /content/spawn_check.py
import multiprocessing

import numpy as np
from qutip import basis, fock_dm, tensor

from quaccatoo import NV, PulsedSim

nv = NV(B0=25, units_B0="mT", N=14)
nv.truncate(mS=1, mI=1)

w1_rf, w1_mwa, w1c = 0.2, 16, 2.14 / 3**0.5
tpi_rf, tpi_mwa, tpi_c = 1 / (2 * w1_rf), 1 / (2 * w1_mwa), 1 / (2 * w1c)
w0_rf, w0_mwa, w0_c = nv.RF_freqs[2], nv.MW_freqs[0], nv.energy_levels[2]
SOL = {"nsteps": 1e6}


def seq(phi, **kwargs):
    s = PulsedSim(nv)
    s.add_pulse(tpi_c, w1c * nv.MW_h1,
                pulse_params={"f_pulse": w0_c, "phi_t": np.pi / 2}, options=SOL)
    s.add_pulse(tpi_rf / 2, w1_rf * nv.RF_h1,
                pulse_params={"f_pulse": w0_rf, "phi_t": phi}, options=SOL)
    s.add_pulse(tpi_mwa, w1_mwa * nv.MW_h1,
                pulse_params={"f_pulse": w0_mwa, "phi_t": np.pi / 2}, options=SOL)
    s.add_pulse(tpi_rf / 2, w1_rf * nv.RF_h1,
                pulse_params={"f_pulse": w0_rf, "phi_t": phi}, options=SOL)
    return s.rho


if __name__ == "__main__":
    multiprocessing.set_start_method("spawn", force=True)  # imitate Windows / macOS
    nv.rho0 = tensor(basis(2, 0) - basis(2, 1), basis(2, 0) - basis(2, 1)).unit()
    nv.observable = [tensor(fock_dm(2, 0), fock_dm(2, 0)),
                     tensor(fock_dm(2, 0), fock_dm(2, 1)),
                     tensor(fock_dm(2, 1), fock_dm(2, 0)),
                     tensor(fock_dm(2, 1), fock_dm(2, 1))]

    phis = np.arange(1.5, 3.3, 0.1)
    for label, map_kw in [("serial (fixed default)", None),
                          ("parallel num_cpus=2 (old behaviour)", {"num_cpus": 2})]:
        sim = PulsedSim(nv)
        sim.run(phis, seq, map_kw=map_kw)
        metric = sim.results[0] ** 2 + sim.results[3] ** 2
        print(f"{label:38s} optimum phi = {phis[np.argmax(metric)]:.2f} rad")

In [ ]:
!LOKY_START_METHOD=spawn python /content/spawn_check.py